<a href="https://colab.research.google.com/github/YogaaAdityaa/uts-sp/blob/main/Soal2_SP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Knowledge Base: Penyakit, Gejala, dan Rules
penyakit = {
    "P1": "Hawar Daun Bakteri",
    "P2": "Blas",
    "P3": "Busuk Batang",
    "P4": "Tungro"
}

gejala = {
    "G1": "Daun menguning",
    "G2": "Bercak coklat pada daun",
    "G3": "Daun mengering",
    "G4": "Batang busuk dan berbau",
    "G5": "Pertumbuhan terhambat",
    "G6": "Daun berlubang",
    "G7": "Daun berkerut",
    "G8": "Tanaman kerdil"
}

# Rules
rules = {
    ("G1",): {"P1": 0.6, "P3": 0.5, "P4": 0.8},
    ("G2",): {"P1": 0.8, "P2": 0.7},
    ("G3",): {"P2": 0.8},
    ("G4",): {"P3": 0.9},
    ("G5",): {"P2": 0.6, "P3": 0.7},
    ("G6",): {"P1": 0.7},
    ("G7",): {"P4": 0.9},
    ("G8",): {"P4": 0.85},
    ("G2", "G6"): {"P1": 0.85},
    ("G1", "G8"): {"P4": 0.9}
}

# Fungsi untuk validasi dan konversi CF user
def validasi_cf(cf_str):
    cf_str = cf_str.lower().strip()
    if cf_str in ["yakin"]:
        return 1.0
    elif cf_str in ["cukup yakin"]:
        return 0.8
    elif cf_str in ["ragu"]:
        return 0.5
    elif cf_str in ["tidak"]:
        return 0.0
    else:
        # Jika input tidak dikenali, gunakan angka
        try:
            cf = float(cf_str)
            if 0 <= cf <= 1:
                return cf
            else:
                raise ValueError
        except ValueError:
            print("Error: Input CF tidak valid. Gunakan kata seperti 'yakin', 'ragu', 'tidak', atau angka 0-1.")
            return None

# Fungsi Forward Chaining dengan perhitungan CF
def forward_chaining(gejala_user, cf_user):
    cf_penyakit = {p: 0.0 for p in penyakit.keys()}
    cf_user_avg = sum(cf_user) / len(cf_user) if cf_user else 0

    for rule_gejala, rule_cf in rules.items():
        # Cek apakah semua gejala di rule ada di gejala_user
        if all(g in gejala_user for g in rule_gejala):
            for penyakit_target, cf_rule in rule_cf.items():
                # Hitung CF(H,E) = CF(user_avg) * CF(rule) untuk kombinasi
                cf_new = cf_user_avg * cf_rule
                # Kombinasi multiple evidence: CF_kombinasi = CF_old + CF_new * (1 - CF_old)
                cf_old = cf_penyakit[penyakit_target]
                cf_penyakit[penyakit_target] = cf_old + cf_new * (1 - cf_old)

    return cf_penyakit

# User Interface
def main():
    print("=== Sistem Pakar Diagnosis Penyakit Tanaman Padi ===")
    print("Gejala tersedia:")
    for kode, desk in gejala.items():
        print(f"  {kode}: {desk}")

    # Input gejala yang dialami
    gejala_user = []
    cf_user = []
    for kode, desk in gejala.items():
        jawab = input(f"Apakah tanaman mengalami {desk}? (y/n): ").lower().strip()
        if jawab == 'y':
            gejala_user.append(kode)
            cf_input = input(f"Seberapa yakin Anda dengan gejala ini? (yakin/cukup yakin/ragu/tidak): ").strip()
            cf_val = validasi_cf(cf_input)
            if cf_val is not None:
                cf_user.append(cf_val)
            else:
                print("Input tidak valid. Menggunakan CF=0.5 sebagai default.")
                cf_user.append(0.5)

    if not gejala_user:
        print("Tidak ada gejala yang dipilih. Program berhenti.")
        return

    # Jalankan Forward Chaining
    hasil_cf = forward_chaining(gejala_user, cf_user)

    # Filter dan ranking penyakit dengan CF > 0
    penyakit_aktif = {p: cf for p, cf in hasil_cf.items() if cf > 0}
    if not penyakit_aktif:
        print("\nTidak ada penyakit yang terdeteksi berdasarkan gejala.")
        return

    ranking = sorted(penyakit_aktif.items(), key=lambda x: x[1], reverse=True)

    # Output: Tampilkan semua penyakit dengan CF > 0 dan ranking
    print("\n=== Hasil Diagnosis ===")
    print("Penyakit dengan CF > 0:")
    for p, cf in penyakit_aktif.items():
        print(f"  {penyakit[p]} ({p}): CF = {cf:.2f}")

    print("\nRanking Penyakit berdasarkan CF tertinggi:")
    for i, (p, cf) in enumerate(ranking, 1):
        print(f"{i}. {penyakit[p]} ({p}): CF = {cf:.2f}")

    # Kesimpulan dan Rekomendasi
    top_penyakit = ranking[0][0]
    top_cf = ranking[0][1]
    print(f"\nKesimpulan: Penyakit paling mungkin adalah {penyakit[top_penyakit]} dengan CF {top_cf:.2f}.")
    print("Rekomendasi: Lakukan pengendalian hama/penyakit sesuai protokol pertanian, seperti penggunaan pestisida organik atau konsultasi dengan ahli pertanian. Pantau tanaman secara berkala untuk mencegah penyebaran.")

# Testing (3 test case)
def test():
    print("\n=== Testing Sistem ===")

    # Test Case 1: Gejala tunggal (G1 -> P1, P3, P4)
    print("Test 1: Gejala ['G1'], CF=[0.8]")
    hasil = forward_chaining(["G1"], [0.8])
    print("Hasil CF:", hasil)

    # Test Case 2: Multiple gejala (G2, G3 -> P1, P2)
    print("Test 2: Gejala ['G2', 'G3'], CF=[0.9, 0.7]")
    hasil = forward_chaining(["G2", "G3"], [0.9, 0.7])
    print("Hasil CF:", hasil)

    # Test Case 3: Kombinasi rules baru (G2, G6 -> P1 dengan rule 13)
    print("Test 3: Gejala ['G2', 'G6'], CF=[1.0, 0.8]")
    hasil = forward_chaining(["G2", "G6"], [1.0, 0.8])
    print("Hasil CF:", hasil)

# Jalankan program
if __name__ == "__main__":
    test()  # Menjalankan test terlebih dahulu
    print()
    main()  # Menjalankan UI utama


=== Testing Sistem ===
Test 1: Gejala ['G1'], CF=[0.8]
Hasil CF: {'P1': 0.48, 'P2': 0.0, 'P3': 0.4, 'P4': 0.6400000000000001}
Test 2: Gejala ['G2', 'G3'], CF=[0.9, 0.7]
Hasil CF: {'P1': 0.6400000000000001, 'P2': 0.8416, 'P3': 0.0, 'P4': 0.0}
Test 3: Gejala ['G2', 'G6'], CF=[1.0, 0.8]
Hasil CF: {'P1': 0.975654, 'P2': 0.63, 'P3': 0.0, 'P4': 0.0}

=== Sistem Pakar Diagnosis Penyakit Tanaman Padi ===
Gejala tersedia:
  G1: Daun menguning
  G2: Bercak coklat pada daun
  G3: Daun mengering
  G4: Batang busuk dan berbau
  G5: Pertumbuhan terhambat
  G6: Daun berlubang
  G7: Daun berkerut
  G8: Tanaman kerdil
Apakah tanaman mengalami Daun menguning? (y/n): y
Seberapa yakin Anda dengan gejala ini? (yakin/cukup yakin/ragu/tidak): yakin
Apakah tanaman mengalami Bercak coklat pada daun? (y/n): n
Apakah tanaman mengalami Daun mengering? (y/n): n
Apakah tanaman mengalami Batang busuk dan berbau? (y/n): n
Apakah tanaman mengalami Pertumbuhan terhambat? (y/n): n
Apakah tanaman mengalami Daun berluban